# Kinyarwanda ASR — Colab GPU pipeline

Runs transcription + KenLM-correction + number conversion on a free Colab GPU
(15-25× faster than local CPU).

## One-time Drive setup
1. In Google Drive, create: `KinyarwandaASR/`
2. Inside it, create subfolders: `assets/`, `audio_in/`, `labels_out/`, `testing_data/`
3. Upload these files into `KinyarwandaASR/assets/`:
   - `kenlm_model.binary`
   - `lm_corpus.txt`
   - `normalize.py`
   - `lm_rescore.py`
   - `kin_numbers.py`  ← new: word-to-digit converter
4. Upload audio files into `KinyarwandaASR/audio_in/` for pseudo-labeling.
5. *(Optional)* For WER evaluation: upload your `testing_data/` folder contents
   (each `.mp3` paired with a `.docx` reference) into `KinyarwandaASR/testing_data/`.

## Each session
- Runtime → Change runtime type → **GPU (T4)**
- Runtime → Run all


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/KinyarwandaASR'
ASSETS_DIR  = f'{PROJECT_DIR}/assets'
AUDIO_DIR   = f'{PROJECT_DIR}/audio_in'
OUTPUT_DIR  = f'{PROJECT_DIR}/labels_out'
TEST_DIR    = f'{PROJECT_DIR}/testing_data'

import os
for d in (PROJECT_DIR, ASSETS_DIR, AUDIO_DIR, OUTPUT_DIR, TEST_DIR):
    os.makedirs(d, exist_ok=True)

REQUIRED = ('kenlm_model.binary', 'lm_corpus.txt', 'normalize.py', 'lm_rescore.py', 'kin_numbers.py')
missing = [f for f in REQUIRED if not os.path.exists(f'{ASSETS_DIR}/{f}')]
if missing:
    print('⚠ Missing from assets/ — upload these before continuing:', missing)
else:
    print('✓ All assets present.')


In [ ]:
!pip install -q faster-whisper kenlm

In [ ]:
# Download the fine-tuned Kinyarwanda Whisper model directly from Hugging Face (fast on Colab's network)
import os
MODEL_DIR = '/content/whisper-kinyarwanda-ct2'
os.makedirs(MODEL_DIR, exist_ok=True)
REPO = 'leophill/whisper-large-v3-turbo-sw-kinyarwanda-ct2'
BASE_URL = f'https://huggingface.co/{REPO}/resolve/main'
for f in ['config.json', 'preprocessor_config.json', 'tokenizer.json', 'vocabulary.json', 'model.bin']:
    dest = f'{MODEL_DIR}/{f}'
    if not os.path.exists(dest):
        !curl -L -C - --retry 50 --retry-delay 5 --retry-all-errors -o "{dest}" "{BASE_URL}/{f}"
print('Model ready at', MODEL_DIR)

In [ ]:
# Make uploaded helper modules importable from Drive assets
import sys
sys.path.insert(0, ASSETS_DIR)

from normalize import normalize_hypothesis
from lm_rescore import correct_words
from kin_numbers import words_to_digits
print('✓ Loaded normalize_hypothesis, correct_words, words_to_digits')


In [ ]:
import json, time
from pathlib import Path
from faster_whisper import WhisperModel

AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg', '.webm')

def output_path_for(audio_path: Path) -> Path:
    safe_stem = ''.join(c if c.isalnum() else '_' for c in audio_path.stem)[:80]
    return Path(OUTPUT_DIR) / f'{safe_stem}.json'

print('Loading model on GPU...')
model = WhisperModel(MODEL_DIR, device='cuda', compute_type='float16')

all_files = sorted(p for p in Path(AUDIO_DIR).iterdir() if p.suffix.lower() in AUDIO_EXTS)
todo = [p for p in all_files if not output_path_for(p).exists()]
print(f'{len(all_files)} audio files in audio_in/, {len(todo)} not yet labeled.')

for i, audio_path in enumerate(todo, 1):
    out_path = output_path_for(audio_path)
    print(f'[{i}/{len(todo)}] {audio_path.name}')
    start = time.time()
    try:
        segments, info = model.transcribe(
            str(audio_path),
            language='sw',
            task='transcribe',
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=300),
            condition_on_previous_text=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            compression_ratio_threshold=2.4,
        )
        seg_records, raw_parts = [], []
        for seg in segments:
            text = seg.text.strip()
            raw_parts.append(text)
            seg_records.append({
                'start': seg.start, 'end': seg.end, 'text': text,
                'avg_logprob': seg.avg_logprob,
                'no_speech_prob': seg.no_speech_prob,
                'compression_ratio': seg.compression_ratio,
            })
        raw_text = ' '.join(raw_parts)
        corrected_text = correct_words(normalize_hypothesis(raw_text))
        elapsed = time.time() - start
        avg_lp = [s['avg_logprob'] for s in seg_records]
        nsp = [s['no_speech_prob'] for s in seg_records]
        record = {
            'source_path': str(audio_path),
            'duration_s': info.duration,
            'transcribe_time_s': elapsed,
            'detected_language': info.language,
            'language_probability': info.language_probability,
            'raw_text': raw_text,
            'corrected_text': corrected_text,
            'segments': seg_records,
            'mean_avg_logprob': sum(avg_lp) / len(avg_lp) if avg_lp else None,
            'mean_no_speech_prob': sum(nsp) / len(nsp) if nsp else None,
            'num_segments': len(seg_records),
        }
        out_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'  duration={record["duration_s"]:.1f}s transcribe_time={record["transcribe_time_s"]:.1f}s '
              f'mean_avg_logprob={record["mean_avg_logprob"]:.3f}')
    except Exception as e:
        print(f'  ERROR: {e}')

print('Done for this session. Outputs in', OUTPUT_DIR)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# QUICK TEST — transcribe one audio file and show full pipeline output
# Set AUDIO_FILE to the name of any file already in audio_in/ or testing_data/
# ─────────────────────────────────────────────────────────────────────────────
import time
from pathlib import Path

AUDIO_FILE = ''   # ← put filename here, e.g. 'ibyakozwe2.mp3'
CONVERT_DIGITS = True   # show number words converted to digits

# Auto-find the file in audio_in/ or testing_data/
audio_path = None
for search_dir in (AUDIO_DIR, TEST_DIR):
    candidate = Path(search_dir) / AUDIO_FILE
    if candidate.exists():
        audio_path = candidate
        break

if not AUDIO_FILE:
    print('Set AUDIO_FILE to the name of the audio file you want to test.')
elif audio_path is None:
    print(f'File not found: {AUDIO_FILE}  (checked audio_in/ and testing_data/)')
else:
    print(f'Transcribing: {audio_path.name}')
    start = time.time()
    segments, info = model.transcribe(
        str(audio_path),
        language='sw',
        task='transcribe',
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
        condition_on_previous_text=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4,
        compression_ratio_threshold=2.4,
    )
    raw_parts = [seg.text.strip() for seg in segments]
    raw_text  = ' '.join(raw_parts)
    normalized = normalize_hypothesis(raw_text)
    corrected  = correct_words(normalized)
    elapsed    = time.time() - start

    print(f'\nDuration: {info.duration:.1f}s  |  Transcribed in: {elapsed:.1f}s  |  Language p={info.language_probability:.2f}')
    print('\n── RAW MODEL OUTPUT ─────────────────────────────────────────────────────')
    print(raw_text)
    print('\n── AFTER normalize.py (proper nouns + French fixes + word splits) ───────')
    print(normalized)
    print('\n── AFTER lm_rescore (KenLM word correction) ────────────────────────────')
    print(corrected)
    if CONVERT_DIGITS:
        print('\n── WITH DIGITS (--digits flag equivalent) ───────────────────────────────')
        print(words_to_digits(corrected))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL: WER / CER evaluation against testing_data/
# Requires: audio + matching .docx reference in KinyarwandaASR/testing_data/
# ─────────────────────────────────────────────────────────────────────────────
import re, time
import docx
import jiwer
from pathlib import Path

AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.flac', '.ogg')

def read_docx(path):
    doc = docx.Document(str(path))
    return ' '.join(p.text.strip() for p in doc.paragraphs if p.text.strip())

def normalize_text(text):
    text = text.lower()
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = re.sub(r'[<>\u00ab\u00bb\u201c\u201d\"\\"\\']', ' ', text)
    text = re.sub(r'[^\w\s\']', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

audio_files = sorted(p for p in Path(TEST_DIR).iterdir() if p.suffix.lower() in AUDIO_EXTS)
pairs = []
for af in audio_files:
    for ext in ('.docx', '.txt'):
        ref = af.with_suffix(ext)
        if ref.exists():
            pairs.append((af, ref))
            break

if not pairs:
    print('No audio+reference pairs found in testing_data/. Upload .mp3 + matching .docx files.')
else:
    print(f'Found {len(pairs)} test pairs. Evaluating...')
    total_wer = total_cer = 0
    for af, ref_path in pairs:
        ref_text = read_docx(ref_path) if ref_path.suffix == '.docx' else ref_path.read_text()
        ref_norm = normalize_text(ref_text)

        segs, info = model.transcribe(
            str(af), language='sw', task='transcribe', beam_size=1,
            vad_filter=True, vad_parameters=dict(min_silence_duration_ms=300),
            condition_on_previous_text=False, repetition_penalty=1.2,
            no_repeat_ngram_size=4, compression_ratio_threshold=2.4,
        )
        hyp = ' '.join(s.text.strip() for s in segs)
        hyp = correct_words(normalize_hypothesis(hyp))
        hyp_norm = normalize_text(hyp)

        wer = jiwer.wer(ref_norm, hyp_norm)
        cer = jiwer.cer(ref_norm, hyp_norm)
        total_wer += wer
        total_cer += cer
        print(f'  {af.stem}: WER={wer:.1%}  CER={cer:.1%}  dur={info.duration:.0f}s')

    n = len(pairs)
    print(f'\n── OVERALL ({n} files) ─────────────────────────────────')
    print(f'  Average WER: {total_wer/n:.2%}')
    print(f'  Average CER: {total_cer/n:.2%}')
